In [1]:
import pandas as pd
import numpy as np
from pptx import Presentation
from pptx.util import Pt
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.util import Inches
import cv2
from pptx.enum.shapes import MSO_SHAPE
from PIL import Image



# 엑셀파일읽어오기
df_raw = pd.read_excel('opda_홍라임.xlsx',sheet_name = 0)

# 값만 있는 파일
df = df_raw.iloc[7:]

# ppt 양식 불러오기
prs = Presentation('koco_frame.pptx')

# 이미지 이름 써주기
img_name = '한가인.jpg'
pic_name = 'error_sample.png'
ioupper = '상악사진.jpg'
iolower = '하악사진.jpg'
iofrontal = '교합정면.jpg'
ioright = '교합우측.jpg'
ioleft = '교합좌측.jpg'


In [2]:
# 양식 입력 함수
def TextFrame(ss,font_name = '맑은 고딕', font_size = Pt(15), font_bold = True, ft_color = True, font_color = RGBColor(68, 84, 116)):
    for line in range(len(ss.text_frame.paragraphs)):
        ss.text_frame.paragraphs[line].font.name = font_name
        ss.text_frame.paragraphs[line].font.size = font_size
        ss.text_frame.paragraphs[line].font.bold = font_bold
        ss.text_frame.paragraphs[line].alignment = PP_ALIGN.CENTER
        if ft_color == True:
            ss.text_frame.paragraphs[line].font.color.rgb = font_color
    return ss

In [3]:
### 슬라이드 양식복사
temp_slide = prs.slides[0]
shape_s = temp_slide.shapes




#ppt table에 ceph 데이터 넣기
for i in range(df.shape[0]):
    shape_s[7].table.cell(row_idx=i,col_idx =1).text = str(df.iloc[i,3])
    TextFrame(shape_s[7].table.cell(row_idx=i,col_idx =1),font_size=Pt(7),font_bold=False,ft_color=False)

In [4]:
#ceph 수치의 공백 없애기(특히 오른쪽)
df['Unnamed: 0'] = df['Unnamed: 0'].str.rstrip()

c:\Users\ok419\Anaconda3\envs\NLP2\lib\site-packages\ipykernel_launcher.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  


In [5]:
# ceph data를 dict로 바꾸기 (변수 활용하기 위해서)
ceph = {}
for key, value in zip(df['Unnamed: 0'],df['Unnamed: 3']):
    ceph[key] = value

# ceph 이름 찾을 때  ceph.keys()

In [6]:
## 수식만들기

import math


#### IAPDI
cosvalue = math.cos(math.radians(ceph['- AB<LOP']))
a = 3.5/4.4 * cosvalue
if ceph['APDI'] >= 81:
    if ceph['PMA'] < 27.5 :
        IAPDI = 95 - 0.5*ceph['PMA']
    elif ceph['PMA'] > 27.5 :
        IAPDI = 81

elif ceph['APDI'] < 81:
    IAPDI = 81-a*(ceph['PMA']-27.5)

def Round_ceph(a):
    return round(a,2)

IAPDI = Round_ceph(IAPDI)
#IAPDI = round(IAPDI,1)

#### HGI,VGI, 2APDL, IODI, VDL, CFD
#round_list = [HGI, VGI, APDL, IODI, VDL, CFD]
HGI = 0.2*((ceph['MBL']-ceph['ACBL'])*2+(ceph['UGA']-50)+0.5*(ceph['PCBA']-64))
VGI = 0.2*((ceph['FHR']-60)*2-(ceph['LGA']-75)+0.5*(ceph['ACBA']-7))
APDL = 0.4*(ceph['APDI'] - IAPDI)
IODI = ((80-0.3*ceph['PMA']-(0.776-0.008*ceph['FMA'])*(ceph['FABA']-80)))
VDL = 0.4849*(ceph['ODI']-IODI)
CFD = ceph['APDI'] + ceph['ODI'] - IAPDI - IODI


HGI = round(HGI,2)
VGI = round(VGI,2)
APDL = round(APDL,2)
IODI = round(IODI,2)
VDL = round(VDL,2)
CFD = round(CFD,2)
#### 

In [7]:
#### 변수 변경하는 란!!!
# 고정변수
fixed_var_dict = {8 : 'C/C & Main problem',
9 : 'MPH:',
11 : f'HGI:{HGI}',
13 : f'VGI:{VGI}',

17 : f'IAPDI:{IAPDI}',
19 : f'2APDL:{APDL*2}',
20 : f'IODI:{IODI}',
22 : f'VDL:{VDL}',

24 : 'CEPH RESULT',

25 : f'CFD:{CFD}',
26 : 'Extraction:'}


# 입력 및 폰트 양식 설정
for k, v in fixed_var_dict.items():
    shape_s[k].text = v
    TextFrame(shape_s[k],font_size=Pt(13),font_bold=True,ft_color = False)

In [8]:
# 제목 인적사항
name = df_raw.iloc[3,1]
age = df_raw.iloc[3,3]
birth = df_raw.iloc[2,3]
gender = f'({df_raw.iloc[4,1][0]})'

#soft_profile 값 정하기
if ceph['FA`B`'] >= 83:
    soft_profile = 'S3'
elif ceph['FA`B`'] >= 79:
    soft_profile = 'S1'
else : soft_profile = 'S2'

#bony_profile 값 정하기
if ceph['FABA'] >= 83:
    bony_profile = 'B3'
elif ceph['FABA'] >= 79:
    bony_profile = 'B1'
else : bony_profile = 'B2'

#denture_profile
if 2*APDL >=2:
    denture_profile = 'D3'
elif -1 <= 2*APDL <2:
    denture_profile = 'D1'
else : denture_profile = 'D2'

#nbt
if ceph['Overbite'] > 3:
    nbt = 'dbt'
elif -2 < ceph['Overbite'] <= 3 :
    nbt = 'nbt'
else : nbt = 'obt'

#skeletal_nbt
if VDL >  1:
    skeletal_nbt = 'dbt'
elif -4 < VDL <= 1:
    skeletal_nbt = 'nbt'
else : skeltal_nbt = 'obt'


# 제목 입력
shape_s[4].text = f'{" ".join([gender,name,age,birth])} \n {soft_profile}.{bony_profile}.{denture_profile}.C1-{nbt}({skeletal_nbt})-RM(Rt)-Fx:Ex-Fx/1-Type IV'

# 양식설정
TextFrame(shape_s[4])
    

In [9]:
from PIL import Image

######### 이미지 리사이즈
# 이미지 불러오기
img = Image.open(img_name)
width, height = img.size
wpercent = 1.6/float(width) 
new_height =  round(float(height)*float(wpercent),1)
img.save(img_name)

######### 사진 집어넣기
left = Inches(2.7)
top = Inches(0.55)
width = Inches(1.6)
height = Inches(new_height)

shape_s.add_picture(img_name,left,top,width,height)

In [10]:
pic_name = 'KakaoTalk_20230113_064749196.jpg'

In [11]:
####2번째 슬라이드 만들기(PSA)

#psa 를 위해서 이미지 객체 만들기
image = cv2.imread(f"{pic_name}", cv2.IMREAD_COLOR)
#psa_img = Image.open(f"{pic_name}")
#print(psa_img.size[1])
height, width, channels = image.shape
print(width)
print(image.shape)
### 이름 나온 곳 검은색으로 칠해주기

x_start = int(width)
y_start = int(height/4)

image = cv2.rectangle(image, (0,0), (x_start, y_start),(0,0,0),-1)


x_start = int(width/3)
y_start = int(height)
image = cv2.rectangle(image, (0,0), (x_start, y_start),(0,0,0),-1)

1704
(841, 1704, 3)


### test

In [12]:
# 초록색 색상 범위 설정
lower_green = (30, 80, 80)
upper_green = (70, 255, 255)


# RGB 에서 HSV 로 색상지정방식 변경
img_hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

#마스크 씌우기
img_mask = cv2.inRange(img_hsv, lower_green, upper_green)

#사진상에서 초록색만 남기는 것(마스크를 씌움)
img_result = cv2.bitwise_and(image, image, mask=img_mask)

In [13]:
# cv2.imshow('tt',img_result)
# cv2.waitKey()
# cv2.destroyAllWindows()

In [14]:
# 두 좌표
d = img_result.nonzero()[0][0]
b = img_result.nonzero()[0][-1]
c = img_result.nonzero()[1][0]
a = img_result.nonzero()[1][-1]

# 하절치점의 좌표는 array 에서 columns 에 해당하는 [1]의 마지막 값이다 [-1] 결국 [1][-1] =a (하절치의 x 좌표)

# 남은 하나 꼭짓점 좌표 구하기
x0 = int((a+c)/2 - (np.sqrt(3)*(d-b))/2)
y0 = int((b+d)/2 + (np.sqrt(3)*(c-a))/2)

x1 = int((a+c)/2 + (np.sqrt(3)*(d-b))/2)
y1 = int((b+d)/2 - (np.sqrt(3)*(c-a))/2)

if x0 > x1:
    x = x0
else:
    x = x1

if y0 < y1:
    y = y0
else:
    y = y1
    
# 좌표 위치 묶어주기 
pts = np.array([[a,b],[c,d],[x,y]],dtype=np.int32)
image = cv2.imread(f"{pic_name}", cv2.IMREAD_COLOR)

# 삼각형 그리기
src = cv2.polylines(image, [pts], isClosed=True, color = (0,255,255))




# 변 길이

lim = int(np.sqrt((a-c)**2+(b-d)**2))

# 원그리기1
src = cv2.circle(image, (x,y), radius=lim, color = (0,255,255))

# 원그리기 2
src = cv2.circle(src, (a,b), radius=lim, color = (0,255,255))

# 결과이미지 저장하기

exp = pic_name.strip().split('.')[0]
cv2.imwrite(f"{exp}_result.png",src)

True

In [15]:
### 슬라이드 양식복사
temp_slide_1 = prs.slides[1]
shape_s_1= temp_slide_1.shapes
# for idx, value in enumerate(shape_s_1):
#         shape_s_1[idx].text = f'{idx},{value.name}'
#         print(idx, value.name)


In [16]:

img_psa = Image.open(f"{exp}_result.png")
if img_psa.size[1]/img_psa.size[0] < 19.05/25.4 :
    w = 10
    width = Inches(w)
    h = w * img_psa.size[1]/img_psa.size[0]
    height = Inches(h)
    left = Inches(0)
    top = Inches(((19.05/2.54)-h)/2)
    
else:
    h = 19.05/2.54
    height = Inches(h)
    w = h * img_psa.size[0]/img_psa.size[1]
    left = Inches((10-w)/2)
    top = Inches(0)
shape_s_1.add_picture(f"{exp}_result.png",left, top, width, height)


In [17]:
### 슬라이드 양식복사
temp_slide_2 = prs.slides[2]
shape_s_2= temp_slide_2.shapes
# for idx, value in enumerate(shape_s_1):
#         shape_s_1[idx].text = f'{idx},{value.name}'
#         print(idx, value.name)


In [18]:
io_list = [ioupper, iolower, iofrontal, ioright, ioleft]
for i in io_list:
    print('1')
    globals()[f'img_{i}'] = Image.open(f'{i}')
    print(globals()[f'img_{i}'].size[0], globals()[f'img_{i}'].size[1])
    globals()[f'img_{i}_width'] = Inches(3)
    globals()[f'img_{i}_height'] = Inches((globals()[f'img_{i}'].size[1]/globals()[f'img_{i}'].size[0])*3)
    print('2')
    
print(globals()[f'img_{ioright}_height'])                              



1
724 485
2
1
725 485
2
1
725 485
2
1
725 485
2
1
725 485
2
1835106


In [19]:
ioupper = '상악사진.jpg'
iolower = '하악사진.jpg'
iofrontal = '교합정면.jpg'
ioright = '교합우측.jpg'
ioleft = '교합좌측.jpg'

#상악사진
img_upper = Image.open(ioupper)
h = 2.2
height = Inches(h)
w = h*img_upper.size[0]/img_upper.size[1]
width = Inches(w)
left = Inches((10-2*w)/2)
top = Inches(1)
shape_s_2.add_picture('상악사진.jpg',left, top, width, height)

#하악사진
img_lower = Image.open(iolower)
left = Inches((10-2*w)/2 + w)
shape_s_2.add_picture('하악사진.jpg',left, top, width, height)

#교합좌측
img_left = Image.open(ioleft)
left = Inches((10-3*w)/2)
top = Inches(1+h)
shape_s_2.add_picture('교합좌측.jpg',left, top, width, height)

#교합정면
img_frontal = Image.open(iofrontal)
left = Inches((10-3*w)/2 + w)
shape_s_2.add_picture('교합정면.jpg',left, top, width, height)

#교합우측
img_right = Image.open(ioright)
left = Inches((10-3*w)/2 + 2*w)
shape_s_2.add_picture('교합우측.jpg',left, top, width, height)




In [20]:
image = cv2.imread(f"{pic_name}", cv2.IMREAD_COLOR)

In [21]:
### 슬라이드 양식복사
temp_slide_3 = prs.slides[3]
shape_s_3= temp_slide_3.shapes
# for idx, value in enumerate(shape_s_1):
#         shape_s_1[idx].text = f'{idx},{value.name}'
#         print(idx, value.name)


In [22]:


w = 10
width = Inches(w)
h = (img_lateral_ceph.size[1]/img_lateral_ceph.size[0])*10
height = Inches(h)
left = Inches(0)
top = Inches(((19.05/2.54)-h)/2)
shape_s_3.add_picture(lateral_ceph ,left, top, width, height)


NameError: name 'img_lateral_ceph' is not defined

In [ ]:
#lateral ceph
lateral_ceph = 'lateral_ceph.jpg'

img_lateral_ceph = cv2.imread(lateral_ceph, cv2.IMREAD_COLOR)
s_x = 500
s_y = 500
color = (0,0,255)
pt1 = (s_x, s_y)
pt2 = (int((s_x +(HGI*100)/4)), int((s_y - (VGI*100)/4)))
img = cv2.arrowedLine(img_lateral_ceph, pt1,pt2, color= (0,0,255))

cv2.imshow('tt',img)
cv2.waitKey()
cv2.destroyAllWindows()

In [ ]:
img

array([[[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 6,  6,  6],
        [17, 17, 17],
        [17, 17, 17]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 3,  3,  3],
        [ 8,  8,  8],
        [ 8,  8,  8]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [ 1,  1,  1],
        [ 2,  2,  2],
        [ 2,  2,  2]],

       ...,

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [21, 21, 21],
        [21, 21, 21],
        [21, 21, 21]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [23, 23, 23],
        [23, 23, 23],
        [23, 23, 23]],

       [[ 0,  0,  0],
        [ 0,  0,  0],
        [ 0,  0,  0],
        ...,
        [24, 24, 24],
        [24, 24, 24],
        [24, 24, 24]]], dtype=uint8)

In [ ]:
# pt1 = 
# cv2.arrowedLine(	img, pt1, pt2, color[, thickness[, line_type[, shift[, tipLength]]]]	)

SyntaxError: invalid syntax (3777102936.py, line 1)

In [ ]:
prs.save('koco_test.pptx')